___
TEXT MINING - SENTENCE EMBEDDINGS
___

# Modern Text Embeddings con Sentence-Transformers

In questo notebook esploreremo gli **embedding moderni** utilizzando la libreria [Sentence-Transformers](https://www.sbert.net/), completamente **open source e gratuita**.

A differenza di Word2Vec e GloVe che generano embedding a livello di parola, i modelli transformer generano **embedding a livello di frase/documento**, catturando meglio il contesto semantico.

## Perché Sentence-Transformers?

- **Gratuito**: Nessuna API key o costi
- **Open Source**: Modelli disponibili su Hugging Face
- **Stato dell'arte**: Basati su architettura Transformer (come BERT, RoBERTa)
- **Facile da usare**: API semplice e intuitiva
- **Multilingue**: Supporta molte lingue incluso l'italiano

## Installazione

Installiamo le librerie necessarie:

In [ ]:
!pip install sentence-transformers -q

## 1. Caricamento del Modello

Utilizziamo `all-MiniLM-L6-v2`, un modello leggero ma efficace:
- **Dimensione embedding**: 384
- **Velocità**: Molto veloce
- **Qualità**: Ottima per la maggior parte degli use case

In [ ]:
from sentence_transformers import SentenceTransformer

# Carichiamo un modello leggero e veloce
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Modello caricato!")
print(f"Dimensione embedding: {model.get_sentence_embedding_dimension()}")

## 2. Generare Embeddings

Vediamo come generare embedding per singole frasi:

In [ ]:
# Generiamo l'embedding di una frase
sentence = "The quick brown fox jumps over the lazy dog."
embedding = model.encode(sentence)

print(f"Frase: '{sentence}'")
print(f"\nShape dell'embedding: {embedding.shape}")
print(f"\nPrimi 10 valori: {embedding[:10]}")

In [ ]:
# Possiamo generare embedding per più frasi contemporaneamente
sentences = [
    "I love machine learning",
    "Deep learning is fascinating",
    "I enjoy eating pizza",
    "The weather is nice today"
]

embeddings = model.encode(sentences)

print(f"Numero di frasi: {len(sentences)}")
print(f"Shape degli embeddings: {embeddings.shape}")

## 3. Calcolo della Similarità Semantica

Gli embedding ci permettono di calcolare quanto due frasi siano semanticamente simili usando la **cosine similarity**:

In [ ]:
from sentence_transformers import util

# Calcoliamo la matrice di similarità
similarity_matrix = util.cos_sim(embeddings, embeddings)

print("Matrice di Similarità:\n")
print(similarity_matrix.numpy().round(3))

In [ ]:
# Visualizziamo la matrice come heatmap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(10, 8))
sns.heatmap(
    similarity_matrix.numpy(), 
    annot=True, 
    fmt='.2f',
    xticklabels=[f"S{i+1}" for i in range(len(sentences))],
    yticklabels=[s[:30] + "..." if len(s) > 30 else s for s in sentences],
    cmap='RdYlGn'
)
plt.title('Matrice di Similarità Semantica')
plt.tight_layout()
plt.show()

Come possiamo vedere:
- "I love machine learning" e "Deep learning is fascinating" hanno alta similarità (entrambe parlano di ML/AI)
- "I enjoy eating pizza" e "The weather is nice today" sono poco correlate alle frasi tecniche

## 4. Semantic Search

Un'applicazione pratica: trovare i documenti più simili a una query.

In [ ]:
# Corpus di documenti
corpus = [
    "Python is a popular programming language for data science.",
    "Machine learning models can predict future outcomes.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by the human brain.",
    "Pizza originated in Naples, Italy.",
    "Natural language processing helps computers understand text.",
    "The Great Wall of China is a famous landmark.",
    "Deep learning has revolutionized computer vision."
]

# Generiamo gli embedding per tutto il corpus
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

print(f"Corpus codificato: {corpus_embeddings.shape}")

In [ ]:
# Query di ricerca
query = "How do artificial intelligence systems work?"
query_embedding = model.encode(query, convert_to_tensor=True)

# Troviamo i documenti più simili
hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)[0]

print(f"Query: '{query}'\n")
print("Risultati più rilevanti:\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [Score: {hit['score']:.3f}] {corpus[hit['corpus_id']]}")

## 5. Visualizzazione degli Embedding con UMAP/t-SNE

Riduciamo la dimensionalità per visualizzare gli embedding in 2D:

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd

# Dataset più ampio per la visualizzazione
texts = [
    # Tech/AI
    "Machine learning is transforming industries",
    "Deep learning uses neural networks",
    "Artificial intelligence can solve complex problems",
    "Python is great for data science",
    "Algorithms process large datasets",
    # Food
    "I love eating Italian pasta",
    "Pizza is my favorite food",
    "Cooking healthy meals is important",
    "Fresh vegetables are nutritious",
    "Chocolate cake is delicious",
    # Travel
    "Paris is a beautiful city",
    "I want to visit Japan",
    "Beach vacations are relaxing",
    "Mountain hiking is adventurous",
    "Exploring new cultures is exciting"
]

categories = ['Tech']*5 + ['Food']*5 + ['Travel']*5

# Generiamo gli embedding
text_embeddings = model.encode(texts)

print(f"Embeddings generati: {text_embeddings.shape}")

In [ ]:
# Riduciamo a 2D con t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(text_embeddings)

# Creiamo un DataFrame per la visualizzazione
df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'category': categories,
    'text': texts
})

# Visualizziamo
plt.figure(figsize=(12, 8))
colors = {'Tech': 'blue', 'Food': 'green', 'Travel': 'red'}

for category in df['category'].unique():
    mask = df['category'] == category
    plt.scatter(df[mask]['x'], df[mask]['y'], 
                c=colors[category], label=category, s=100, alpha=0.7)

# Aggiungiamo le etichette
for idx, row in df.iterrows():
    plt.annotate(row['text'][:25] + '...', 
                 (row['x'], row['y']),
                 fontsize=8, alpha=0.8)

plt.legend()
plt.title('Visualizzazione t-SNE degli Embedding')
plt.xlabel('Dimensione 1')
plt.ylabel('Dimensione 2')
plt.tight_layout()
plt.show()

## 6. Modelli Multilingue

Sentence-Transformers offre anche modelli multilingue che supportano l'italiano:

In [ ]:
# Carichiamo un modello multilingue
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Testi in diverse lingue con lo stesso significato
texts_multilingual = [
    "I love artificial intelligence",      # Inglese
    "Amo l'intelligenza artificiale",      # Italiano
    "J'adore l'intelligence artificielle", # Francese
    "Me encanta la inteligencia artificial", # Spagnolo
    "Ich liebe künstliche Intelligenz",    # Tedesco
    "The weather is beautiful today",      # Frase diversa (inglese)
    "Oggi il tempo è bellissimo"           # Frase diversa (italiano)
]

# Generiamo gli embedding
multilingual_embeddings = multilingual_model.encode(texts_multilingual)

# Calcoliamo la similarità
multilingual_similarity = util.cos_sim(multilingual_embeddings, multilingual_embeddings)

print("Similarità tra frasi multilingue:\n")
for i, text_i in enumerate(texts_multilingual):
    for j, text_j in enumerate(texts_multilingual):
        if i < j:
            sim = multilingual_similarity[i][j].item()
            print(f"[{sim:.3f}] '{text_i[:40]}...' <-> '{text_j[:40]}...'")

In [ ]:
# Visualizziamo la matrice di similarità
plt.figure(figsize=(12, 10))
labels = [t[:20] + "..." for t in texts_multilingual]

sns.heatmap(
    multilingual_similarity.numpy(),
    annot=True,
    fmt='.2f',
    xticklabels=labels,
    yticklabels=labels,
    cmap='RdYlGn'
)
plt.title('Similarità Cross-Linguistica')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Osservazione**: Le frasi che parlano di "intelligenza artificiale" in diverse lingue hanno alta similarità tra loro, dimostrando che il modello cattura il significato semantico indipendentemente dalla lingua!

## 7. Confronto tra Diversi Modelli

Proviamo diversi modelli disponibili gratuitamente:

In [ ]:
# Lista di modelli da confrontare (tutti gratuiti!)
model_names = [
    'all-MiniLM-L6-v2',           # Veloce e leggero (384 dim)
    'all-mpnet-base-v2',          # Più accurato (768 dim)
    'paraphrase-MiniLM-L6-v2',    # Ottimo per parafrasare
]

test_sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",  # Parafrasi della prima
    "Dogs are loyal animals"        # Frase diversa
]

print("Confronto tra modelli:\n")
print(f"Frase 1: {test_sentences[0]}")
print(f"Frase 2: {test_sentences[1]} (parafrasi)")
print(f"Frase 3: {test_sentences[2]} (diversa)\n")

for model_name in model_names:
    test_model = SentenceTransformer(model_name)
    test_embeddings = test_model.encode(test_sentences)
    
    sim_1_2 = util.cos_sim(test_embeddings[0], test_embeddings[1]).item()
    sim_1_3 = util.cos_sim(test_embeddings[0], test_embeddings[2]).item()
    
    print(f"Modello: {model_name}")
    print(f"  Similarità (1-2 parafrasi): {sim_1_2:.3f}")
    print(f"  Similarità (1-3 diversa):   {sim_1_3:.3f}")
    print(f"  Dimensione: {test_model.get_sentence_embedding_dimension()}\n")

## 8. Clustering con Embeddings

Usiamo gli embedding per raggruppare automaticamente testi simili:

In [ ]:
from sklearn.cluster import KMeans

# Dataset per il clustering
documents = [
    # Sport
    "The football match ended in a draw",
    "Tennis players compete at Wimbledon",
    "Basketball requires teamwork and skill",
    # Tecnologia
    "Smartphones have changed communication",
    "Artificial intelligence is advancing rapidly",
    "Cloud computing enables remote work",
    # Natura
    "Forests are essential for biodiversity",
    "Ocean pollution threatens marine life",
    "Climate change affects weather patterns"
]

# Generiamo gli embedding
doc_embeddings = model.encode(documents)

# Applichiamo K-Means clustering
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(doc_embeddings)

# Visualizziamo i risultati
print("Risultati del Clustering:\n")
for cluster_id in range(n_clusters):
    print(f"\nCluster {cluster_id}:")
    for doc, cluster in zip(documents, clusters):
        if cluster == cluster_id:
            print(f"  - {doc}")

## 9. Uso Pratico: Duplicate Detection

Troviamo frasi duplicate o quasi-duplicate in un dataset:

In [ ]:
# Dataset con potenziali duplicati
corpus_with_duplicates = [
    "How do I reset my password?",
    "I forgot my password, how can I reset it?",
    "What are your business hours?",
    "When is your store open?",
    "I need help with my order",
    "Can you assist me with a purchase issue?",
    "The product arrived damaged",
    "My item was broken when delivered"
]

# Generiamo gli embedding
dup_embeddings = model.encode(corpus_with_duplicates, convert_to_tensor=True)

# Troviamo coppie simili (soglia > 0.7)
threshold = 0.7
similarity_matrix = util.cos_sim(dup_embeddings, dup_embeddings)

print(f"Potenziali duplicati (similarità > {threshold}):\n")
for i in range(len(corpus_with_duplicates)):
    for j in range(i+1, len(corpus_with_duplicates)):
        sim = similarity_matrix[i][j].item()
        if sim > threshold:
            print(f"[{sim:.3f}]")
            print(f"  1: {corpus_with_duplicates[i]}")
            print(f"  2: {corpus_with_duplicates[j]}\n")

## Riepilogo

In questo notebook abbiamo visto:

1. **Sentence-Transformers**: Libreria open source per embedding di frasi
2. **Generazione di embedding**: Come convertire testo in vettori numerici
3. **Similarità semantica**: Calcolo della cosine similarity tra embedding
4. **Semantic search**: Ricerca di documenti rilevanti
5. **Visualizzazione**: t-SNE per visualizzare embedding in 2D
6. **Modelli multilingue**: Supporto per italiano e altre lingue
7. **Clustering**: Raggruppamento automatico di testi
8. **Duplicate detection**: Trovare testi simili/duplicati

### Modelli consigliati (tutti gratuiti!):

| Modello | Dimensione | Velocità | Uso |
|---------|------------|----------|-----|
| `all-MiniLM-L6-v2` | 384 | Veloce | General purpose |
| `all-mpnet-base-v2` | 768 | Media | Massima qualità |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Media | Multilingue |

### Risorse:
- [Sentence-Transformers Documentation](https://www.sbert.net/)
- [Hugging Face Model Hub](https://huggingface.co/models?library=sentence-transformers)
- [Pretrained Models](https://www.sbert.net/docs/pretrained_models.html)